# 01 データ探索（EDA）

Stack Exchange API から取得した質問データ（12,000件）と
タグ統計データ（42件）の全体像を把握する。


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13

df = pd.read_parquet('../data/processed/questions.parquet')
tags_df = pd.read_parquet('../data/processed/tags.parquet')
print(f"質問データ: {df.shape[0]:,}行 × {df.shape[1]}列")
print(f"タグ統計:   {tags_df.shape[0]}件")

## 1. 基本統計

In [ ]:
# データ型と欠損値の確認
print("=== dtypes ===")
print(df.dtypes[['tags', 'title', 'score', 'answer_count', 'view_count',
                  'creation_date', 'is_answered', 'language', 'year']])
print()
print("=== 欠損値 ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# スコア・回答数・閲覧数の記述統計
df[['score', 'answer_count', 'view_count']].describe().round(1)

## 2. 言語別・年別 質問数分布

In [ ]:
pivot = df.groupby(['year', 'language']).size().unstack()
ax = pivot.plot(kind='bar', figsize=(13, 5), width=0.75)
ax.set_title('言語別 質問数推移（サンプル: 各年300件）')
ax.set_xlabel('年')
ax.set_ylabel('質問数')
ax.legend(title='言語', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../data/analysis/fig_question_count_by_year.png', dpi=120, bbox_inches='tight')
plt.show()
print("※ 各年300件の均等サンプリングのため件数は同一。実際の質問数は tags_df で確認。")

## 3. タグ統計：言語別の実際の質問総数

In [ ]:
# 各言語のメインタグの総質問数を棒グラフで表示
main_tags = tags_df[tags_df.apply(
    lambda r: r['name'] in ['python','javascript','java','go'], axis=1
)].copy()

colors = {'python': '#3776AB', 'javascript': '#F7DF1E',
          'java': '#ED8B00', 'go': '#00ADD8'}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(main_tags['name'], main_tags['count'] / 1e6,
              color=[colors.get(n, '#888') for n in main_tags['name']])
ax.set_title('Stack Overflow 言語別 累積質問数（2024年時点）')
ax.set_ylabel('質問数（百万件）')
ax.bar_label(bars, labels=[f"{v:.2f}M" for v in main_tags['count'] / 1e6], padding=3)
plt.tight_layout()
plt.savefig('../data/analysis/fig_total_questions.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. スコア分布・回答率

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# スコア分布（外れ値を除いた箱ひげ図）
score_data = [df[df['language']==lang]['score'].clip(-5, 20)
              for lang in ['python', 'javascript', 'java', 'go']]
axes[0].boxplot(score_data, labels=['Python', 'JavaScript', 'Java', 'Go'],
                patch_artist=True,
                boxprops=dict(facecolor='#4f8ef7', alpha=0.6))
axes[0].set_title('スコア分布（外れ値除外）')
axes[0].set_ylabel('スコア')
axes[0].axhline(0, color='gray', linewidth=0.5)

# 回答率
answer_rate = df.groupby('language')['is_answered'].mean().sort_values()
bars = axes[1].barh(answer_rate.index, answer_rate.values * 100,
                    color='#4f8ef7', alpha=0.8)
axes[1].set_title('言語別 回答率（%）')
axes[1].set_xlabel('回答率（%）')
axes[1].bar_label(bars, labels=[f"{v:.1f}%" for v in answer_rate.values * 100],
                  padding=3)
plt.tight_layout()
plt.savefig('../data/analysis/fig_score_answer_rate.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. 言語別 共起タグ トップ10

In [ ]:
# tags 列がリスト型であることを確認して explode
sample = df.copy()
if isinstance(sample['tags'].iloc[0], list):
    tags_exploded = sample.explode('tags')
else:
    tags_exploded = sample.copy()
    tags_exploded['tags'] = tags_exploded['tags'].str.strip("[]").str.replace("'","").str.split(", ")
    tags_exploded = tags_exploded.explode('tags')

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, lang in zip(axes.flatten(), ['python', 'javascript', 'java', 'go']):
    sub = tags_exploded[tags_exploded['language'] == lang]
    # メインタグ自身を除いた共起タグ
    exclude = {'python', 'javascript', 'java', 'go', 'python-3.x'}
    top = (sub[~sub['tags'].isin(exclude)]['tags']
           .value_counts().head(10))
    ax.barh(top.index[::-1], top.values[::-1], color='#4f8ef7', alpha=0.8)
    ax.set_title(f'{lang.capitalize()} の共起タグ トップ10')
    ax.set_xlabel('出現件数')

plt.tight_layout()
plt.savefig('../data/analysis/fig_cooccurrence_tags.png', dpi=120, bbox_inches='tight')
plt.show()

## EDA まとめ

| 観点 | 発見 |
|---|---|
| 累積質問数 | JavaScript > Python > Java > Go の順（Go は新興言語で少ない） |
| 回答率 | 4言語とも 60〜80%。Java が最も高い傾向 |
| スコア分布 | 中央値は0〜2点付近。高スコアは外れ値的に少ない |
| 共起タグ | Python は pandas/numpy が突出。JS は react/node.js が中心 |

次のステップ（Day 5〜）でこれらの傾向をより詳細に分析する。
